# Ejemplo Procesamiento de imágenes

**Curso**: INF3841 - Recuperación de Información  
**Profesor**: Juan Manuel Barrios  
**Fecha**: 03 de agosto de 2026  

En este ejemplo se muestra como abrir una imagen y aplicar un filtro usando OpenCV.

Requiere:
```
pip install opencv-contrib-python matplotlib PySide6
```

## Definir funciones auxiliares

Se usa un diálogo en QT para seleccionar la imagen a procesar.

In [ ]:
import sys
import os
import numpy
import cv2
import matplotlib.pyplot as plt
from PySide6.QtWidgets import QApplication, QFileDialog


def mostrar_imagen(window_name, imagen):
    MIN_WIDTH, MAX_WIDTH = 200, 1000
    MIN_HEIGHT, MAX_HEIGHT = 200, 800
    if imagen.shape[0] > MAX_HEIGHT or imagen.shape[1] > MAX_WIDTH:
        # si es muy grande, reducir tamaño
        fh = MAX_HEIGHT / imagen.shape[0]
        fw = MAX_WIDTH / imagen.shape[1]
        escala = min(fh, fw)
        imagen = cv2.resize(imagen, (0, 0), fx=escala, fy=escala, interpolation=cv2.INTER_CUBIC)
    elif imagen.shape[0] < MIN_HEIGHT or imagen.shape[1] < MIN_WIDTH:
        # si es muy pequeña, aumentar tamaño
        fh = MIN_HEIGHT / imagen.shape[0]
        fw = MIN_WIDTH / imagen.shape[1]
        escala = max(fh, fw)
        imagen = cv2.resize(imagen, (0, 0), fx=escala, fy=escala, interpolation=cv2.INTER_NEAREST)
    # mostrar en pantalla
    cv2.imshow(window_name, imagen)


def ui_select_filenames():
    # mostrar un dialogo de seleccionar un archivo
    app = QApplication(list())
    options = QFileDialog.Options()
    files, _ = QFileDialog.getOpenFileNames(
        None, "Imagenes", ".", "Imagenes (*.jpg *.png)", options=options
    )
    app.shutdown()
    return files


def abrir_imagen(filename):
    imagen_color = cv2.imread(filename, cv2.IMREAD_COLOR)
    if imagen_color is None:
        raise Exception("error abriendo {}".format(filename))
    return imagen_color


def histograma_gris(imagen_8bits):
    bins = range(0, 257, 1)
    xticks = list(range(0, 256, 64))
    xticks.append(255)
    plt.hist(imagen_8bits.reshape(-1), bins=bins, density=True)
    plt.xticks(xticks)
    plt.xlabel("grises")
    plt.ylabel("cantidad")
    plt.title("Histograma de intensidades")
    plt.show()


print(
    "Usando Python {}.{}.{} con OpenCV {}".format(
        sys.version_info.major,
        sys.version_info.minor,
        sys.version_info.micro,
        cv2.__version__,
    )
)

## Ejemplo 1 - Histograma y Ecualización

In [ ]:
def ejemplo(filename):
    imagen_color = abrir_imagen(filename)
    imagen_gris = cv2.cvtColor(imagen_color, cv2.COLOR_BGR2GRAY)
    imagen_eq = cv2.equalizeHist(imagen_gris)
    mostrar_imagen(filename + " (gris)", imagen_gris)
    mostrar_imagen(filename + " (eq)", imagen_eq)
    histograma_gris(imagen_gris)
    histograma_gris(imagen_eq)


# abre un selector de archivos
filenames = ui_select_filenames()

if len(filenames) > 0:
    for filename in filenames:
        ejemplo(filename)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

print("FIN")

## Ejemplo 2 - Binarización con OTSU

In [ ]:
def ejemplo(filename):
    imagen_color = abrir_imagen(filename)
    imagen_gris = cv2.cvtColor(imagen_color, cv2.COLOR_BGR2GRAY)
    threshold, imagen_bin = cv2.threshold(
        imagen_gris, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU
    )
    mostrar_imagen(filename, imagen_color)
    mostrar_imagen(filename + " (gris)", imagen_gris)
    mostrar_imagen(filename + " (bin)", imagen_bin)
    print("{} size={} threshold={}".format(filename, imagen_color.shape, threshold))
    histograma_gris(imagen_gris)
    histograma_gris(imagen_bin)


# abre un selector de archivos
filenames = ui_select_filenames()

if len(filenames) > 0:
    for filename in filenames:
        ejemplo(filename)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

print("FIN")

## Ejemplo 3 - Convolución

En OpenCV normalmente se usa la función **filter2d** [[en este link]](https://docs.opencv.org/5.0/main_modules/imgproc_filter.html#filter2d).

Notar que `cv2.filter2d()` calcula la correlación y no convolución (ver la documentación). Si se desea calcular la convolución, antes se debe reflejar el kernel con `kernel2 = cv2.flip(kernel, flipCode=-1)` y si el kernel es de tamaño par se debe mover el centro (ver documentación).

Con respecto al tratamiento del borde, por defecto usa bordes tipo `BORDER_REFLECT_101` que la documentación describe así:

`gfedcb | `**`abcdefgh`**` | gfedcba`

es decir, para una imagen que en una fila tiene pixeles `abcdefgh`, crea pixeles antes y después como si estuvieran en un espejo. Se pueden ver otros parámetros para el borde en: https://docs.opencv.org/5.0/main_modules/core_array.html#bordertypes  
En general, tiene un efecto pequeño qué hacer en el borde porque afecta unos pocos pixeles, sin embargo, se vuelve relevante si las imágenes son muy pequeñas o cuando contienen algún patrón especial.

Alternativamente, la librería SciPy tiene la función `scipy.signal.convolve2d()`  que también implementa la convolución [[ver documentación]](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.convolve2d.html).

In [ ]:
def ejemplo(filename):
    imagen_color = abrir_imagen(filename)
    imagen_gris = cv2.cvtColor(imagen_color, cv2.COLOR_BGR2GRAY)
    imagen_gaussian = cv2.GaussianBlur(imagen_gris, (9, 9), 0)
    imagen_median = cv2.medianBlur(imagen_gris, 9)
    kernel = numpy.array([[1, 1, 1], [1, 1, 1], [1, 1, 1]]) / 9
    # una forma alternativa para probar kernels más grandes:
    # kernel = numpy.ones((15, 15), numpy.float32) / (15*15)
    imagen_conv = cv2.filter2D(imagen_gris, -1, kernel)
    mostrar_imagen("gris: " + filename, imagen_gris)
    mostrar_imagen("gaussian: " + filename, imagen_gaussian)
    mostrar_imagen("median: " + filename, imagen_median)
    mostrar_imagen("convolucion: " + filename, imagen_conv)


filenames = ui_select_filenames()

if len(filenames) > 0:
    for filename in filenames:
        ejemplo(filename)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

print("FIN")

Más detalles y ejemplos se pueden ver en la página de OpenCV: https://docs.opencv.org/5.0/tutorials/imgproc/imgtrans/filter_2d/filter_2d.html